In [5]:
import pandas as pd
import glob

In [ ]:
files = glob.glob("data/cars/*.csv")

df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

df.to_csv("data/raw/uk_used_cars.csv", index=False)

In [11]:
df = pd.read_csv("../data/raw/uk_used_cars.csv")

C:\Users\harut\AppData\Local\Temp\ipykernel_23544\1134224167.py:1: DtypeWarning: Columns (0: price, 1: mileage, 2: fuel type, 3: engine size, 4: mileage2, 5: fuel type2, 6: engine size2, 7: reference) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/uk_used_cars.csv")


In [12]:
df.head()

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,tax(£),fuel type,engine size,mileage2,fuel type2,engine size2,reference
0,A1,2017.0,12500,Manual,15735,Petrol,150.0,55.4,1.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A6,2016.0,16500,Automatic,36203,Diesel,20.0,64.2,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A1,2016.0,11000,Manual,29946,Petrol,30.0,55.4,1.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A4,2017.0,16800,Automatic,25952,Diesel,145.0,67.3,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A3,2019.0,17300,Manual,1998,Petrol,145.0,49.6,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 118150 entries, 0 to 118149
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   model         117995 non-null  str    
 1   year          117903 non-null  float64
 2   price         117995 non-null  object 
 3   transmission  117995 non-null  str    
 4   mileage       117077 non-null  object 
 5   fuelType      108540 non-null  str    
 6   tax           94327 non-null   float64
 7   mpg           99187 non-null   float64
 8   engineSize    108540 non-null  float64
 9   tax(£)        4860 non-null    float64
 10  fuel type     3517 non-null    str    
 11  engine size   9345 non-null    str    
 12  mileage2      9399 non-null    str    
 13  fuel type2    8537 non-null    str    
 14  engine size2  8537 non-null    str    
 15  reference     9455 non-null    str    
dtypes: float64(5), object(2), str(9)
memory usage: 14.4+ MB


In [7]:
df.isna().sum()

model              155
year               247
price              155
transmission       155
mileage           1073
fuelType          9610
tax              23823
mpg              18963
engineSize        9610
tax(£)          113290
fuel type       114633
engine size     108805
mileage2        108751
fuel type2      109613
engine size2    109613
reference       108695
dtype: int64

In [8]:
len(df[df["tax(£)"].isna()]) + len(df[~df["tax(£)"].isna()]) == len(df)

True

In [9]:
df[['tax(£)', 'tax']].notna().all()

tax(£)    False
tax       False
dtype: bool

In [10]:
print(df['fuelType'].unique())
print(df['fuel type'].unique())

<StringArray>
['Petrol', 'Diesel', 'Hybrid', 'Other', 'Electric', nan]
Length: 6, dtype: str
<StringArray>
[     nan, 'Diesel', 'Petrol',     '29',     '34',     '37',     '31',
     '32',     '38',     '33',     '28',     '25',     '36',     '26',
     '30',     '44',     '35',     '43',     '27',     '39',     '40',
     '24',     '46',     '45',     '48', 'Hybrid',     '47',     '11',
     '14',     '12',     '10',     '13',     '15',     '16',     '22',
      '8',     '21',      '9',     '20',      '7',     '19',     '18',
     '17',      '6']
Length: 44, dtype: str


In [11]:
df['fuelType'] = df['fuelType'].combine_first(df['fuel type'])
df['fuelType'] = df['fuelType'].combine_first(df['fuel type2'])

df['engineSize'] = df['engineSize'].combine_first(df['engine size'])
df['engineSize'] = df['engineSize'].combine_first(df['engine size2'])

df['mileage'] = df['mileage'].combine_first(df['mileage2'])

df['tax'] = df['tax'].combine_first(df['tax(£)'])

In [12]:
df.isna().sum()

model              155
year               247
price              155
transmission       155
mileage            155
fuelType           155
tax              18963
mpg              18963
engineSize         155
tax(£)          113290
fuel type       114633
engine size     108805
mileage2        108751
fuel type2      109613
engine size2    109613
reference       108695
dtype: int64

In [13]:
df.drop(
    columns=[
        'fuel type',
        'fuel type2',
        'engine size',
        'engine size2',
        'mileage2',
        'tax(£)'
    ],
    inplace=True
)

In [14]:
df.isna().sum().sort_values(ascending=False)

reference       108695
tax              18963
mpg              18963
year               247
model              155
price              155
fuelType           155
mileage            155
transmission       155
engineSize         155
dtype: int64

In [15]:
df['reference'].nunique()

9455

In [13]:
df['reference'].value_counts()

reference
/ad/25017331    1
/ad/25043746    1
/ad/25142894    1
/ad/24942816    1
/ad/24913660    1
               ..
/ad/25149519    1
/ad/25149523    1
/ad/25149532    1
/ad/25149535    1
/ad/25149536    1
Name: count, Length: 9455, dtype: int64

In [16]:
missing_pct = (
    df.isnull()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

print(missing_pct[missing_pct > 0])

reference       91.997461
tax             16.049937
mpg             16.049937
year             0.209056
model            0.131189
price            0.131189
fuelType         0.131189
mileage          0.131189
transmission     0.131189
engineSize       0.131189
dtype: float64


In [17]:
print(df.columns.tolist())
print(df.shape)
print(df.info())

['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax', 'mpg', 'engineSize', 'reference']
(118150, 10)
<class 'pandas.DataFrame'>
RangeIndex: 118150 entries, 0 to 118149
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   model         117995 non-null  str    
 1   year          117903 non-null  float64
 2   price         117995 non-null  object 
 3   transmission  117995 non-null  str    
 4   mileage       117995 non-null  object 
 5   fuelType      117995 non-null  str    
 6   tax           99187 non-null   float64
 7   mpg           99187 non-null   float64
 8   engineSize    117995 non-null  object 
 9   reference     9455 non-null    str    
dtypes: float64(3), object(3), str(4)
memory usage: 9.0+ MB
None


In [18]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 2427


In [19]:
df = df.drop(columns="reference")
df = df.drop_duplicates()
print(df.shape)

(114961, 9)


In [20]:
df.isna().sum()

model               1
year               89
price               1
transmission        1
mileage             1
fuelType            1
tax             17249
mpg             17249
engineSize          1
dtype: int64

In [21]:
df[df['tax'].isna()][['model','year','price','fuelType','engineSize']].head()

,model,year,price,fuelType,engineSize
21449,C Class,2020.0,30495,Diesel,2.0
21450,C Class,2020.0,29989,Petrol,1.5
21451,C Class,2020.0,37899,Diesel,2.0
21452,C Class,2019.0,30399,Diesel,2.0
21453,C Class,2019.0,29899,Diesel,2.0


In [22]:
df[df['mpg'].isna()][['model','year','price','fuelType','engineSize']].head()

,model,year,price,fuelType,engineSize
21449,C Class,2020.0,30495,Diesel,2.0
21450,C Class,2020.0,29989,Petrol,1.5
21451,C Class,2020.0,37899,Diesel,2.0
21452,C Class,2019.0,30399,Diesel,2.0
21453,C Class,2019.0,29899,Diesel,2.0


In [23]:
df['tax'] = (
    df.groupby(['fuelType','engineSize'])['tax']
    .transform(lambda x: x.fillna(x.median()))
)

In [24]:
df['tax'].isna().sum()

np.int64(8696)

In [25]:
df['mpg'] = (
    df.groupby(['fuelType','engineSize'])['mpg']
    .transform(lambda x: x.fillna(x.median()))
)

In [26]:
df.isnull().sum()

model              1
year              89
price              1
transmission       1
mileage            1
fuelType           1
tax             8696
mpg             8696
engineSize         1
dtype: int64

In [27]:
(df['tax'].isna() == df['mpg'].isna()).value_counts()

True    114961
Name: count, dtype: int64

In [28]:
df['tax'] = df['tax'].fillna(df['tax'].median())
df['mpg'] = df['mpg'].fillna(df['mpg'].median())

In [29]:
df.isnull().sum()

model            1
year            89
price            1
transmission     1
mileage          1
fuelType         1
tax              0
mpg              0
engineSize       1
dtype: int64

In [30]:
df = df.dropna()

In [31]:
df.info()

<class 'pandas.DataFrame'>
Index: 114872 entries, 0 to 118149
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   model         114872 non-null  str    
 1   year          114872 non-null  float64
 2   price         114872 non-null  object 
 3   transmission  114872 non-null  str    
 4   mileage       114872 non-null  object 
 5   fuelType      114872 non-null  str    
 6   tax           114872 non-null  float64
 7   mpg           114872 non-null  float64
 8   engineSize    114872 non-null  object 
dtypes: float64(3), object(3), str(3)
memory usage: 8.8+ MB


In [32]:
df.head()

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
0,A1,2017.0,12500,Manual,15735,Petrol,150.0,55.4,1.4
1,A6,2016.0,16500,Automatic,36203,Diesel,20.0,64.2,2.0
2,A1,2016.0,11000,Manual,29946,Petrol,30.0,55.4,1.4
3,A4,2017.0,16800,Automatic,25952,Diesel,145.0,67.3,2.0
4,A3,2019.0,17300,Manual,1998,Petrol,145.0,49.6,1.0


In [33]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['mileage'] = pd.to_numeric(df['mileage'], errors='coerce')

In [34]:
df['engineSize'].unique()[:10]

array([1.4, 2.0, 1.0, 3.0, 1.6, 1.8, 1.5, 4.0, 2.5, 1.2], dtype=object)

In [35]:
df['engineSize'] = pd.to_numeric(df['engineSize'], errors='coerce')

In [36]:
df.isnull().sum()

model              0
year               0
price           8605
transmission       0
mileage         8371
fuelType           0
tax                0
mpg                0
engineSize      8262
dtype: int64

In [37]:
df[df.price.isna()]

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
79751,C Class,2020.0,NaN,Automatic,NaN,Diesel,145.0,55.4,2.0
79752,C Class,2020.0,NaN,Automatic,NaN,Petrol,145.0,55.4,1.5
79753,C Class,2020.0,NaN,Automatic,500.0,Diesel,145.0,55.4,2.0
79754,C Class,2019.0,NaN,Automatic,NaN,Diesel,145.0,55.4,2.0
79755,C Class,2019.0,NaN,Automatic,NaN,Diesel,145.0,55.4,2.0
...,...,...,...,...,...,...,...,...,...
89320,Focus,2014.0,NaN,Manual,NaN,16,145.0,55.4,NaN
89321,Focus,2019.0,NaN,Automatic,NaN,Petrol,145.0,55.4,NaN
89322,Focus,2006.0,NaN,Manual,NaN,15,145.0,55.4,NaN
89339,Focus,2019.0,NaN,Automatic,NaN,Petrol,145.0,55.4,1.0


In [38]:
df = df.dropna(subset=['price'])

In [39]:
df.isna().sum()

model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
dtype: int64

In [40]:
df.info()

<class 'pandas.DataFrame'>
Index: 106267 entries, 0 to 118149
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   model         106267 non-null  str    
 1   year          106267 non-null  float64
 2   price         106267 non-null  float64
 3   transmission  106267 non-null  str    
 4   mileage       106267 non-null  float64
 5   fuelType      106267 non-null  str    
 6   tax           106267 non-null  float64
 7   mpg           106267 non-null  float64
 8   engineSize    106267 non-null  float64
dtypes: float64(6), str(3)
memory usage: 8.1 MB


In [41]:
df['year'] = df['year'].astype(int)

In [42]:
df.duplicated().sum()

np.int64(995)

In [43]:
df[df.duplicated(keep=False)].sort_values(
    by=['model','year']
).head(20)

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
24970,C Class,2011,9495.0,Automatic,39000.0,Petrol,160.0,43.5,1.8
65556,C Class,2011,9495.0,Automatic,39000.0,Petrol,160.0,43.5,1.8
25060,C Class,2012,8495.0,Automatic,51564.0,Petrol,160.0,43.5,1.8
65621,C Class,2012,8495.0,Automatic,51564.0,Petrol,160.0,43.5,1.8
21498,C Class,2013,23990.0,Automatic,44000.0,Petrol,570.0,23.5,6.2
24428,C Class,2013,21999.0,Automatic,41866.0,Petrol,570.0,23.5,6.2
25307,C Class,2013,22948.0,Automatic,39000.0,Petrol,570.0,23.5,6.2
63320,C Class,2013,23990.0,Automatic,44000.0,Petrol,570.0,23.5,6.2
63628,C Class,2013,21999.0,Automatic,41866.0,Petrol,570.0,23.5,6.2
65624,C Class,2013,22948.0,Automatic,39000.0,Petrol,570.0,23.5,6.2


In [44]:
df = df.drop_duplicates()

In [45]:
df.describe()

,year,price,mileage,tax,mpg,engineSize
count,105272.000000,105272.000000,105272.000000,105272.000000,105272.000000,105272.000000
mean,2017.060424,16855.637349,23319.795919,121.603465,55.392230,1.668787
std,2.136080,9811.373674,21172.239327,61.430391,16.002049,0.557057
min,1970.000000,450.000000,1.000000,0.000000,0.300000,0.000000
25%,2016.000000,10090.000000,7773.000000,125.000000,47.100000,1.300000
50%,2017.000000,14500.000000,17678.500000,145.000000,55.400000,1.600000
75%,2019.000000,20890.000000,32598.500000,145.000000,62.800000,2.000000
max,2060.000000,159999.000000,323000.000000,580.000000,470.800000,6.600000


In [46]:
df[df['year'] < 1990]

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
65699,M Class,1970,24999.0,Automatic,14000.0,Diesel,305.0,39.2,0.0
100198,Zafira,1970,10495.0,Manual,37357.0,Petrol,200.0,42.2,1.4


In [47]:
df[df['year'] > 2025]

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
48528,Fiesta,2060,6495.0,Automatic,54807.0,Petrol,205.0,42.8,1.4


In [48]:
df = df[(df['year'] >= 1990) & (df['year'] <= 2026)]

In [49]:
df[df['mpg'] > 150]

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
2550,Q7,2017,32998.0,Semi-Auto,66477.0,Hybrid,140.0,156.9,3.0
3106,A3,2015,14981.0,Semi-Auto,28294.0,Hybrid,0.0,188.3,1.4
4261,A3,2015,17990.0,Automatic,14000.0,Hybrid,0.0,188.3,1.4
4493,A3,2014,15490.0,Automatic,24597.0,Hybrid,0.0,188.3,1.4
4559,A3,2015,16000.0,Semi-Auto,48954.0,Hybrid,0.0,176.6,1.4
...,...,...,...,...,...,...,...,...,...
109180,Passat,2017,23792.0,Semi-Auto,28068.0,Hybrid,135.0,166.0,1.4
109205,Passat,2017,25500.0,Semi-Auto,22400.0,Hybrid,135.0,166.0,1.4
109306,Passat,2018,25999.0,Automatic,32656.0,Hybrid,135.0,166.0,1.4
109343,Passat,2018,25000.0,Automatic,8848.0,Hybrid,135.0,166.0,1.4


In [50]:
df = df[df['mpg'] < 150]

In [51]:
df[df['engineSize'] == 0]

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
7505,Q5,2019,44790.0,Automatic,5886.0,Petrol,135.0,117.7,0.0
7506,Q3,2019,32788.0,Automatic,1500.0,Diesel,145.0,47.1,0.0
7516,Q3,2020,29944.0,Manual,1500.0,Petrol,145.0,40.9,0.0
7517,Q3,2020,33333.0,Automatic,1500.0,Diesel,145.0,47.1,0.0
7518,Q3,2020,29944.0,Automatic,1500.0,Petrol,145.0,32.5,0.0
...,...,...,...,...,...,...,...,...,...
114641,Tiguan,2016,15300.0,Manual,38398.0,Diesel,145.0,53.3,0.0
114648,Tiguan,2018,24000.0,Automatic,22200.0,Diesel,145.0,47.9,0.0
115814,Up,2017,8500.0,Manual,20324.0,Petrol,20.0,64.2,0.0
115872,Up,2017,8000.0,Manual,24444.0,Petrol,20.0,60.1,0.0


In [52]:
df = df[df['engineSize'] > 0]

In [53]:
df[df['price'] > 100000]

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
1646,R8,2019,129000.0,Semi-Auto,4000.0,Petrol,145.0,21.4,5.2
2255,R8,2020,137995.0,Semi-Auto,70.0,Petrol,145.0,21.1,5.2
3359,R8,2019,125000.0,Automatic,100.0,Petrol,145.0,24.1,5.2
3367,R8,2019,135000.0,Automatic,1000.0,Petrol,145.0,32.8,5.2
3939,R8,2019,112990.0,Automatic,8175.0,Petrol,145.0,21.6,5.2
4179,R8,2019,137500.0,Semi-Auto,10.0,Petrol,150.0,21.4,5.2
4400,RS6,2020,102544.0,Semi-Auto,2000.0,Petrol,145.0,22.1,4.0
4742,R8,2019,117990.0,Automatic,11936.0,Petrol,145.0,21.4,5.2
4783,R8,2020,145000.0,Semi-Auto,2000.0,Petrol,145.0,21.1,5.2
4925,R8,2019,125000.0,Semi-Auto,500.0,Petrol,145.0,21.4,5.2


In [54]:
df[df['price'].isin([99999, 111111, 123456, 12345])]

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
14306,2 Series,2015,123456.0,Semi-Auto,33419.0,Diesel,20.0,68.9,2.0


In [55]:
df = df[df['price'] != 123456]

In [56]:
df.shape

(104753, 9)

In [315]:
df.describe()

,year,price,mileage,tax,mpg,engineSize
count,104753.000000,104753.000000,104753.000000,104753.000000,104753.000000,104753.000000
mean,2017.059005,16833.834678,23326.468913,121.662005,54.953001,1.673239
std,2.122976,9795.996622,21174.852815,61.403311,12.276512,0.551338
min,1991.000000,450.000000,1.000000,0.000000,0.300000,1.000000
25%,2016.000000,10030.000000,7780.000000,125.000000,47.100000,1.300000
50%,2017.000000,14500.000000,17673.000000,145.000000,55.400000,1.600000
75%,2019.000000,20850.000000,32614.000000,145.000000,62.800000,2.000000
max,2020.000000,159999.000000,323000.000000,580.000000,148.700000,6.600000


In [316]:
df.isna().sum()

model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
dtype: int64

In [318]:
df.to_csv("data/used_cars_clean.csv", index=False)